In [ ]:
import tensorflow as tf
import tensorflow_hub as hub
import numpy as np
from google.colab import files

print('Loading YAMNet from TensorFlow Hub...')
# YAMNet is a pre-trained acoustic model that can recognize 521 different sounds
yamnet_model = hub.load('https://tfhub.dev/google/yamnet/1')


Loading YAMNet from TensorFlow Hub...


In [ ]:
class PerfectTransferModel(tf.Module):
    def __init__(self, yamnet):
        super().__init__()
        self.yamnet = yamnet

        print('Engineering Custom Neural Mapping Matrix...')
        # We create a matrix to map 521 classes down to 3
        # Class 0: Background
        # Class 1: Car Horn
        # Class 2: Siren
        w = np.zeros((521, 3), dtype=np.float32)

        # We identify all YAMNet classes related to Car Horns
        car_horn_idx = [102, 302, 304, 312] # Honk, Car Alarm, Vehicle Horn, Air Horn

        # We identify all YAMNet classes related to Emergency Sirens & Alarms
        siren_idx = [11, 317, 318, 319, 382, 390, 393, 394, 475] # Screaming, Police, Ambulance, Fire, Smoke Detector, Beeps

        # Populate the Custom Keras-style routing matrix
        for i in range(521):
            if i in car_horn_idx:
                w[i, 1] = 1.0 # Route to Output 1
            elif i in siren_idx:
                w[i, 2] = 1.0 # Route to Output 2
            else:
                w[i, 0] = 1.0 # Route everything else to Output 0

        self.mapping_weights = tf.constant(w, dtype=tf.float32)

    # This function tells TensorFlow exactly what audio shape to expect from the Android microphone
    @tf.function(input_signature=[tf.TensorSpec(shape=[15600], dtype=tf.float32, name='waveform')])
    def __call__(self, waveform):
        # 1. YAMNet extracts 521 acoustic probabilities
        scores, embeddings, spectrogram = self.yamnet(waveform)

        # 2. Our custom layer distills those 521 probabilities into our 3 target classes
        custom_scores = tf.matmul(scores, self.mapping_weights)

        return custom_scores

# Instantiate our custom architecture
perfect_model = PerfectTransferModel(yamnet_model)
print("Custom Architecture Assembled Successfully!")


Engineering Custom Neural Mapping Matrix...
Custom Architecture Assembled Successfully!


In [ ]:
print('Converting Architecture to TFLite (Maximum Precision)...')

# Get the concrete function for export
concrete_func = perfect_model.__call__.get_concrete_function()
converter = tf.lite.TFLiteConverter.from_concrete_functions([concrete_func])

# CRITICAL: We must freeze all resource memory variables so the Android interpreter can run it natively!
converter.experimental_enable_resource_variables = False

# Convert the model (We purposely avoid quantization to preserve maximum acoustic sensitivity)
perfect_tflite_model = converter.convert()

output_path = 'custom_safehear.tflite'
with open(output_path, 'wb') as f:
    f.write(perfect_tflite_model)

print(f'EXPORT SUCCESSFUL! Downloading {output_path} to your computer...')
files.download(output_path)


Converting Architecture to TFLite (Maximum Precision)...
EXPORT SUCCESSFUL! Downloading custom_safehear.tflite to your computer...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import os
import time


In [ ]:
# Academic Metrics Evaluator
def evaluate_tflite_model(model_path, model_name):
    print(f"\n--- Metrics for {model_name} ---")

    # 1. Measure File Size
    size_mb = os.path.getsize(model_path) / (1024 * 1024)
    print(f"Model File Size: {size_mb:.2f} MB")

    # Load the TFLite model into the Android-style Interpreter
    interpreter = tf.lite.Interpreter(model_path=model_path)
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()

    # Create dummy audio data simulating a phone microphone
    input_shape = input_details[0]['shape']
    dummy_audio = np.random.uniform(-1, 1, size=input_shape).astype(np.float32)

    # Warm-up run
    interpreter.set_tensor(input_details[0]['index'], dummy_audio)
    interpreter.invoke()

    # 2. Measure Inference Speed (Latency) over 100 runs
    num_runs = 100
    start_time = time.time()
    for _ in range(num_runs):
        interpreter.set_tensor(input_details[0]['index'], dummy_audio)
        interpreter.invoke()
    end_time = time.time()

    avg_latency_ms = ((end_time - start_time) / num_runs) * 1000
    print(f"Average Inference Latency: {avg_latency_ms:.2f} milliseconds")
    print("-----------------------------------")

# Evaluate our Original (Float32) Model
evaluate_tflite_model('custom_safehear.tflite', 'Original Float32 Model')



--- Metrics for Original Float32 Model ---
Model File Size: 14.32 MB


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


Average Inference Latency: 3.09 milliseconds
-----------------------------------


In [ ]:
print('Exporting Quantized Architecture (INT8 Optimization)...')
converter_quant = tf.lite.TFLiteConverter.from_concrete_functions([concrete_func])
converter_quant.experimental_enable_resource_variables = False

# THIS is the command that activates Quantization Compression!
converter_quant.optimizations = [tf.lite.Optimize.DEFAULT]

quantized_tflite_model = converter_quant.convert()

quant_path = 'quantized_safehear.tflite'
with open(quant_path, 'wb') as f:
    f.write(quantized_tflite_model)

# Evaluate the Quantized Model to compare!
evaluate_tflite_model(quant_path, 'Quantized (INT8) Model')

# Download both models to your computer
files.download('custom_safehear.tflite')
files.download('quantized_safehear.tflite')


Exporting Quantized Architecture (INT8 Optimization)...

--- Metrics for Quantized (INT8) Model ---
Model File Size: 3.77 MB
Average Inference Latency: 3.78 milliseconds
-----------------------------------


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>